# WHW13 starter notebook: the Fourier series of a square wave, and the overshoot

The square wave is

$$f(x) = \begin{cases} +1 & 0 < x < \pi \\ -1 & -\pi < x < 0 \end{cases},$$

extended with period $2\pi$. Here's the problem:

1. Work out its Fourier series **by hand**.
2. **Before plotting**, predict: as you add more terms, does the largest error
   between the partial sum and the square wave shrink to zero, and whereabouts
   on the interval does the worst error sit? Write the prediction down.
3. Plot the partial sums with 1, 3, 9, and 49 nonzero terms against the exact
   square wave.
4. Make a second plot: the size of the **overshoot** near $x = 0$ as a function of
   the number of terms.
5. Explain in two or three sentences what your second plot says about
   convergence at a jump discontinuity, and say what the series converges to
   *exactly at* $x = 0$. Note one thing you tried that did not work.

We haven't added up Fourier series in Python yet, so this notebook shows you
the NumPy on a *different* function first, the triangle wave, then leaves the
square wave to you.

## 0. Setup

Same setup as always: matplotlib for plotting, sympy for symbolic manipulation,
and numpy for numerical python calculations.

In [ ]:
%matplotlib inline
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import *
init_printing()

## 1. By hand

Work out the Fourier series of the square wave on paper. Which coefficients
are zero, and why? Write your $a_n$ and $b_n$ here. (You can check your by-hand
series against the book: it's printed in the reading, Felder p467, Discovery
Exercise 9.5.1.)

*(your answer here)*

## 2. Predict first (before you plot anything)

1. As you add more and more terms, does the **largest** error between the
   partial sum and the square wave shrink to zero?
2. Whereabouts on the interval does the worst error sit?

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 3. Partial sums in NumPy (demo: the triangle wave)

Demo function: the triangle wave, $g(x) = |x|$ on $-\pi < x < \pi$, extended with
period $2\pi$. It's even, so it's all cosines:

$$g(x) = \frac{\pi}{2} - \frac{4}{\pi}\left(\cos x + \frac{\cos 3x}{3^2} + \frac{\cos 5x}{5^2} + \cdots\right),$$

that is, $a_0/2 = \pi/2$ and $a_n = -\dfrac{4}{\pi n^2}$ for odd $n$, zero for even $n$.

Here's a function that adds up the first `n_terms` **nonzero** cosine terms
(plus the constant), at every $x$ in an array at once:

- `total` starts as the constant term, $\pi/2$, at every $x$. `np.ones(len(x_vals))`
  is an array of ones the same length as `x_vals`.
- Only odd $n$ have nonzero coefficients, so the loop counts `j = 0, 1, 2, ...`
  and uses `n = 2 * j + 1`, which is `1, 3, 5, ...`.
- Each pass adds one term, at every $x$ at once (element-wise, as in the WHW06 starter).

In [ ]:
def triangle_partial_sum(x_vals, n_terms):
    total = (np.pi / 2) * np.ones(len(x_vals))
    for j in range(n_terms):
        n = 2 * j + 1
        total = total + (-4 / (np.pi * n**2)) * np.cos(n * x_vals)
    return total

In [ ]:
x_plot = np.linspace(-np.pi, np.pi, 4001)

plt.figure(figsize=(9, 4))
plt.plot(x_plot, np.abs(x_plot), 'k', linewidth=3, label='triangle wave |x|')
for n_terms in [1, 2, 5]:
    plt.plot(x_plot, triangle_partial_sum(x_plot, n_terms), label=f'{n_terms} nonzero cosine terms')
plt.xlabel('x')
plt.legend()
plt.show()

## 4. One number per term count, collected in a loop (demo)

The question "how does the error depend on the number of terms?" means: for
each number of terms, compute the partial sum, boil the error down to one
number, save it, and plot all the saved numbers at the end.

- `np.abs(partial - exact)` is the error at every $x$; `np.max(...)` picks out the
  biggest entry of an array: the worst error anywhere on the grid.
- `term_counts = np.arange(1, 101)` is `1, 2, ..., 100` (`arange` stops one short of its end).
- `worst_errors = np.zeros(len(term_counts))` makes one empty slot per term count,
  and the loop fills slot `i` with the answer for `term_counts[i]`, as in the
  WHW04 starter.

In [ ]:
term_counts = np.arange(1, 101)
worst_errors = np.zeros(len(term_counts))
for i in range(len(term_counts)):
    partial = triangle_partial_sum(x_plot, term_counts[i])
    worst_errors[i] = np.max(np.abs(partial - np.abs(x_plot)))

plt.figure(figsize=(7, 4))
plt.plot(term_counts, worst_errors, '.-')
plt.xlabel('number of nonzero terms')
plt.ylabel('worst error anywhere')
plt.title('triangle wave')
plt.ylim(bottom=0)
plt.show()

For the triangle wave, the worst error marches down toward zero as we add
terms: the partial sums converge to $|x|$ *uniformly*, everywhere at once. The
triangle wave is continuous (it has corners, but no jumps).

## 5. What to look for: the Gibbs phenomenon

The square wave is different: it **jumps**, from $-1$ to $+1$ at $x = 0$ (and back
at $x = \pm\pi$). Something happens next to a jump that never happens for a
continuous function, and it's what your second plot is about.

Right next to a jump, every partial sum **overshoots**: it climbs past the top
of the jump, wiggles, and settles down. As you add terms, the overshoot gets
squeezed closer to the jump -- it gets *narrower* -- but its *height* does
**not** go to zero. It levels off at about 9% of the size of the jump. This is
called the **Gibbs phenomenon**. (Felder covers it in an online section we
didn't assign; you don't need it to do this problem.)

That's a claim. Your second plot tests it. Two practical notes:

- We'll measure the overshoot as (the biggest value the partial sum reaches on
  $0 < x < \pi$) minus 1, the height of the square wave there.
- The overshoot bump gets narrow: with $K$ nonzero terms its peak sits near
  $x \approx \pi/(2K)$. Your grid has to be fine enough to land on it, so section 7
  uses 20001 points on $[0, \pi]$.

## 6. Your turn: partial sums of the square wave

Copy of `triangle_partial_sum`, renamed. Change the marked lines so it adds up
your square-wave series from section 1:

1. What's the constant term for the square wave? That's where `total` starts.
2. Change the line inside the loop to add your $n$-th term (a sine this time).

The cell after it plots 1, 3, 9 and 49 nonzero terms against the exact square
wave. `np.sign(x)` is $+1$ for positive $x$, $-1$ for negative $x$ (and 0 at $x = 0$),
which is exactly our square wave on $-\pi < x < \pi$.

In [ ]:
def my_partial_sum(x_vals, n_terms):
    # Change the line below.
    total = (np.pi / 2) * np.ones(len(x_vals))
    for j in range(n_terms):
        n = 2 * j + 1
        # Change the line below.
        total = total + (-4 / (np.pi * n**2)) * np.cos(n * x_vals)
    return total

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x_plot, np.sign(x_plot), 'k', linewidth=3, label='square wave')
for n_terms in [1, 3, 9, 49]:
    plt.plot(x_plot, my_partial_sum(x_plot, n_terms), label=f'{n_terms} nonzero terms')
plt.xlabel('x')
plt.legend()
plt.show()

## 7. Your turn: the overshoot as a function of the number of terms

Same loop as section 4, on a fine grid on $[0, \pi]$. Change the marked line so
that `overshoots[i]` is the overshoot from section 5: the biggest value the
partial sum reaches on the grid, minus 1. (As handed out, it computes the
triangle-wave error, so it runs.)

In [ ]:
x_fine = np.linspace(0, np.pi, 20001)
term_counts = np.arange(1, 101)
overshoots = np.zeros(len(term_counts))
for i in range(len(term_counts)):
    partial = my_partial_sum(x_fine, term_counts[i])
    # Change the line below.
    overshoots[i] = np.max(np.abs(partial - np.abs(x_fine)))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(term_counts, overshoots, '.-')
plt.xlabel('number of nonzero terms')
plt.ylabel('overshoot near x = 0')
plt.ylim(bottom=0)
plt.show()
print('overshoot with 1, 3, 9, 49, 100 terms:', overshoots[[0, 2, 8, 48, 99]])

## 8. What you found

**(a)** In two or three sentences: what does your second plot say about
convergence at a jump discontinuity? Compare with your prediction from section 2.

*(your answer here)*

**(b)** What does the series converge to *exactly at* $x = 0$? Why?

*(your answer here)*

**(c)** One thing you tried that did not work (a grid too coarse to catch the
peak, a cosine where you wanted a sine, an even $n$ that snuck in, ...).

*(your answer here)*